# triple2vec — Skip-gram Training for SNOMED CT Concept Relationships

Trains a modified word2vec skip-gram model on a SNOMED CT context dictionary  
`{ center_AUI: [context_AUI, ...] }` using noise-contrastive negative sampling.

**Workflow**
1. Mount Google Drive and point `DATA_PATH` to your JSON file
2. Adjust hyper-parameters in the **Configuration** cell
3. Run all cells — training progress, loss curve, and missing-concept predictions are shown inline
4. The trained model and embeddings are saved back to Drive

## 1 · Install / upgrade dependencies

In [ ]:
# PyTorch ships pre-installed on Colab; tqdm and matplotlib do too.
# Uncomment only if you need a specific version.
# !pip install -q torch torchvision tqdm matplotlib

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
# Confirm the mount worked
print(os.listdir("/content/drive/MyDrive")[:10])

## 3 · Configuration

Edit these variables before running the rest of the notebook.

In [ ]:
# ── Data paths (Google Drive) ────────────────────────────────────────────────
# Path to your context_dict JSON file on Drive
DATA_PATH        = "/content/drive/MyDrive/triple2vec/context_dict.json"

# Where to save the trained model and embeddings
OUTPUT_DIR       = "/content/drive/MyDrive/triple2vec/outputs"
MODEL_FILE       = "triple2vec_model.pt"       # full model checkpoint
EMBEDDING_FILE   = "triple2vec_embeddings.npy" # center embeddings only (numpy)

# ── Model hyper-parameters ───────────────────────────────────────────────────
EMBED_DIM        = 128   # embedding dimensionality
NEG_SAMPLES      = 10    # negative samples per positive pair

# ── Training hyper-parameters ────────────────────────────────────────────────
EPOCHS           = 100
BATCH_SIZE       = 2048
LEARNING_RATE    = 1e-3
LR_MIN           = 1e-5  # cosine annealing floor
NUM_WORKERS      = 2     # DataLoader workers (keep ≤ 2 on Colab)

# ── Checkpointing ────────────────────────────────────────────────────────────
CHECKPOINT_EVERY = 10    # save a checkpoint every N epochs

# ── Prediction ───────────────────────────────────────────────────────────────
PREDICT_TOP_K       = 20  # missing concepts to rank per center
PREDICT_MAX_CENTERS = 10  # how many centers to show after training

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42

# ── Device ───────────────────────────────────────────────────────────────────
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Create output directory if it doesn't exist
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4 · Imports

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Seed everything
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 5 · Load data & build vocabulary

In [ ]:
class Vocabulary:
    """Bidirectional mapping between AUI strings and integer indices."""

    def __init__(self, context_dict: dict):
        auids = set(context_dict.keys())
        for neighbors in context_dict.values():
            auids.update(neighbors)

        self.aui2idx: dict[str, int] = {aui: i for i, aui in enumerate(sorted(auids))}
        self.idx2aui: list[str] = sorted(auids)
        self.size = len(self.aui2idx)

    def __len__(self):
        return self.size

    def encode(self, aui: str) -> int:
        return self.aui2idx[aui]

    def decode(self, idx: int) -> str:
        return self.idx2aui[idx]


# ── Load ────────────────────────────────────────────────────────────────────
print(f"Loading: {DATA_PATH}")
with open(DATA_PATH) as f:
    context_dict = json.load(f)

vocab = Vocabulary(context_dict)

print(f"  Center concepts : {len(context_dict):>10,}")
print(f"  Vocabulary size : {vocab.size:>10,} unique AUIs")

# Quick sanity check
sample_center = next(iter(context_dict))
print(f"  Sample center   : {sample_center}  →  {context_dict[sample_center][:3]} …")

## 6 · Dataset

In [ ]:
class SkipGramDataset(Dataset):
    """
    Flattens context_dict into (center_idx, context_idx) pairs.
    Negative samples are drawn on-the-fly inside the model forward pass.
    """

    def __init__(self, context_dict: dict, vocab: Vocabulary):
        self.pairs: list[tuple[int, int]] = []
        skipped_centers = 0
        skipped_contexts = 0

        for center_aui, context_auis in context_dict.items():
            if center_aui not in vocab.aui2idx:
                skipped_centers += 1
                continue
            c_idx = vocab.encode(center_aui)
            for ctx_aui in context_auis:
                if ctx_aui in vocab.aui2idx:
                    self.pairs.append((c_idx, vocab.encode(ctx_aui)))
                else:
                    skipped_contexts += 1

        if skipped_centers or skipped_contexts:
            print(f"  Skipped {skipped_centers} centers, {skipped_contexts} context AUIs (not in vocab)")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        center, context = self.pairs[idx]
        return torch.tensor(center, dtype=torch.long), torch.tensor(context, dtype=torch.long)


dataset = SkipGramDataset(context_dict, vocab)
print(f"Total (center, context) training pairs: {len(dataset):,}")

## 7 · Model

In [ ]:
class SkipGram(nn.Module):
    """
    Skip-gram with two embedding tables (center / context) and
    noise-contrastive negative sampling loss.

    Loss per pair:
        L = -log σ(v_c · v_pos) - Σ_k log σ(-v_c · v_neg_k)
    """

    def __init__(self, vocab_size: int, embed_dim: int, neg_samples: int):
        super().__init__()
        self.vocab_size  = vocab_size
        self.neg_samples = neg_samples

        self.center_emb  = nn.Embedding(vocab_size, embed_dim)
        self.context_emb = nn.Embedding(vocab_size, embed_dim)

        # Standard word2vec initialisation
        nn.init.uniform_(self.center_emb.weight,  -0.5 / embed_dim, 0.5 / embed_dim)
        nn.init.zeros_(self.context_emb.weight)

    def forward(self, center: torch.Tensor, positive: torch.Tensor) -> torch.Tensor:
        """
        center:   (B,)  — center concept indices
        positive: (B,)  — known context concept indices
        Returns scalar mean loss.
        """
        B = center.size(0)

        c_emb   = self.center_emb(center)    # (B, D)
        pos_emb = self.context_emb(positive) # (B, D)

        # Positive term
        pos_score = (c_emb * pos_emb).sum(dim=1)                        # (B,)
        pos_loss  = -torch.nn.functional.logsigmoid(pos_score)          # (B,)

        # Negative term — random draws from the full vocabulary
        neg_idx   = torch.randint(0, self.vocab_size,
                                  (B, self.neg_samples), device=center.device)  # (B, K)
        neg_emb   = self.context_emb(neg_idx)                           # (B, K, D)
        neg_score = torch.bmm(neg_emb, c_emb.unsqueeze(2)).squeeze(2)   # (B, K)
        neg_loss  = -torch.nn.functional.logsigmoid(-neg_score).sum(1)  # (B,)

        return (pos_loss + neg_loss).mean()

    # ── Inference ────────────────────────────────────────────────────────────

    @torch.no_grad()
    def score_all_contexts(self, center_idx: int) -> torch.Tensor:
        """σ(center_emb · context_emb_matrix) for every concept. Returns (V,)."""
        c_emb  = self.center_emb.weight[center_idx]   # (D,)
        scores = self.context_emb.weight @ c_emb      # (V,)
        return torch.sigmoid(scores)

    @torch.no_grad()
    def predict_missing(
        self,
        center_idx: int,
        known_context_indices: list[int],
        top_k: int = 20,
    ) -> list[tuple[int, float]]:
        """
        Rank all concepts not already known as context for center_idx.
        Returns list of (concept_idx, score) sorted descending.
        """
        scores = self.score_all_contexts(center_idx).cpu()
        suppress = set(known_context_indices) | {center_idx}
        for idx in suppress:
            scores[idx] = -1.0
        top_v, top_i = torch.topk(scores, min(top_k, self.vocab_size))
        return list(zip(top_i.tolist(), top_v.tolist()))

    @torch.no_grad()
    def get_center_embeddings(self) -> np.ndarray:
        return self.center_emb.weight.cpu().numpy()


model = SkipGram(vocab.size, EMBED_DIM, NEG_SAMPLES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}  "
      f"(vocab={vocab.size:,}, embed_dim={EMBED_DIM}, neg_samples={NEG_SAMPLES})")

## 8 · Train

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LR_MIN
)

checkpoint_path = Path(OUTPUT_DIR) / "triple2vec_checkpoint.pt"
epoch_losses: list[float] = []

epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Epochs", unit="epoch")

for epoch in epoch_bar:
    model.train()
    running_loss = 0.0

    for center, positive in loader:
        center   = center.to(DEVICE)
        positive = positive.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        loss = model(center, positive)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * center.size(0)

    scheduler.step()
    avg_loss = running_loss / len(dataset)
    epoch_losses.append(avg_loss)

    epoch_bar.set_postfix(
        loss=f"{avg_loss:.5f}",
        lr=f"{scheduler.get_last_lr()[0]:.2e}"
    )

    # Periodic checkpoint to Drive
    if epoch % CHECKPOINT_EVERY == 0:
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "epoch_losses": epoch_losses,
            },
            checkpoint_path,
        )
        tqdm.write(f"  ✓ Checkpoint saved at epoch {epoch} → {checkpoint_path}")

print(f"\nTraining complete. Final loss: {epoch_losses[-1]:.6f}")

## 9 · Loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(epoch_losses) + 1), epoch_losses, linewidth=1.5, color="steelblue")
ax.set_xlabel("Epoch")
ax.set_ylabel("Avg NCE loss")
ax.set_title("triple2vec training loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()

plot_path = Path(OUTPUT_DIR) / "loss_curve.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Loss curve saved → {plot_path}")

## 10 · Save model & embeddings to Drive

In [ ]:
model_path = Path(OUTPUT_DIR) / MODEL_FILE
torch.save(
    {
        "model_state": model.state_dict(),
        "vocab": {"aui2idx": vocab.aui2idx, "idx2aui": vocab.idx2aui},
        "config": {
            "vocab_size": vocab.size,
            "embed_dim":  EMBED_DIM,
            "neg_samples": NEG_SAMPLES,
        },
        "epoch_losses": epoch_losses,
    },
    model_path,
)
print(f"Model saved  → {model_path}")

emb_path = Path(OUTPUT_DIR) / EMBEDDING_FILE
np.save(emb_path, model.get_center_embeddings())
print(f"Embeddings saved → {emb_path}  shape={model.get_center_embeddings().shape}")

## 11 · Predict missing context concepts

For each center concept, the model scores every concept in the vocabulary,  
suppresses known contexts, and returns the top-K candidates most likely to be missing.

In [ ]:
model.eval()
model.cpu()  # run inference on CPU to avoid transfer overhead for single queries

centers_to_show = list(context_dict.keys())[:PREDICT_MAX_CENTERS]

all_predictions: dict[str, list[tuple[str, float]]] = {}

for center_aui in centers_to_show:
    if center_aui not in vocab.aui2idx:
        continue

    center_idx   = vocab.encode(center_aui)
    known_ctx    = [vocab.encode(a) for a in context_dict[center_aui] if a in vocab.aui2idx]
    raw_preds    = model.predict_missing(center_idx, known_ctx, top_k=PREDICT_TOP_K)

    decoded = [(vocab.decode(idx), score) for idx, score in raw_preds]
    all_predictions[center_aui] = decoded

    print(f"\n{'─'*60}")
    print(f"Center : {center_aui}   (known contexts: {len(known_ctx)})")
    print(f"{'Rank':<6} {'AUI':<14} {'Score'}")
    print(f"{'─'*6} {'─'*14} {'─'*6}")
    for rank, (aui, score) in enumerate(decoded, 1):
        print(f"{rank:<6} {aui:<14} {score:.4f}")

## 12 · Export predictions to JSON (optional)

In [ ]:
# Serialisable format: { center_aui: [{"aui": ..., "score": ...}, ...] }
export = {
    center: [{"aui": aui, "score": round(score, 6)} for aui, score in preds]
    for center, preds in all_predictions.items()
}

pred_path = Path(OUTPUT_DIR) / "predicted_missing_contexts.json"
with open(pred_path, "w") as f:
    json.dump(export, f, indent=2)

print(f"Predictions saved → {pred_path}")

## 13 · Reload model from Drive (optional — run in a new session)

Use this cell if you want to run inference without re-training.

In [ ]:
# ── Reload ───────────────────────────────────────────────────────────────────
checkpoint = torch.load(
    Path(OUTPUT_DIR) / MODEL_FILE,
    map_location="cpu",
)

cfg = checkpoint["config"]
loaded_model = SkipGram(cfg["vocab_size"], cfg["embed_dim"], cfg["neg_samples"])
loaded_model.load_state_dict(checkpoint["model_state"])
loaded_model.eval()

# Restore vocabulary
class _ReloadedVocab:
    def __init__(self, saved):
        self.aui2idx = saved["aui2idx"]
        self.idx2aui = saved["idx2aui"]
        self.size    = len(self.aui2idx)
    def encode(self, aui):  return self.aui2idx[aui]
    def decode(self, idx):  return self.idx2aui[idx]

loaded_vocab = _ReloadedVocab(checkpoint["vocab"])
print(f"Loaded model — vocab {loaded_vocab.size:,}, embed_dim {cfg['embed_dim']}")

# ── Example query ────────────────────────────────────────────────────────────
query_center = "A2941532"   # ← change to any AUI in your vocabulary

# Load your context_dict to know existing edges (or pass an empty list)
# with open(DATA_PATH) as f:
#     context_dict = json.load(f)

c_idx     = loaded_vocab.encode(query_center)
known_ctx = []  # fill with encoded known contexts if available
preds     = loaded_model.predict_missing(c_idx, known_ctx, top_k=PREDICT_TOP_K)

print(f"\nTop-{PREDICT_TOP_K} predicted missing contexts for {query_center}:")
for rank, (idx, score) in enumerate(preds, 1):
    print(f"  {rank:>3}. {loaded_vocab.decode(idx)}  {score:.4f}")